# Lab 4 - Cryptographic Protection

Interactive notebook. For each part: **run** the demo, **implement** the function in the *Your code* cell, then **run the check cell** below it (it should print `PASS`).

Full theory and the research tasks are in `README.md`. Part B needs `pip install cryptography`; the rest is the standard library.

## Part A - Hashing, encoding & randomness  ·  A04:2025

Encoding (Base64) is reversible with no key. Hashing is a one-way fingerprint. Keys/tokens/salts must come from a **cryptographic** RNG.

In [ ]:
from demo_hashing import encoding_is_not_encryption, broken_vs_strong_hash, predictable_vs_unpredictable
encoding_is_not_encryption(); broken_vs_strong_hash(); predictable_vs_unpredictable()

**A.3 Your code**

In [ ]:
import hashlib, base64, secrets

def sha256_hex(data):
    # TODO: hashlib.sha256(...).hexdigest()
    raise NotImplementedError

def secure_token(n_bytes=32):
    # TODO: secrets.token_urlsafe(n_bytes)
    raise NotImplementedError

def is_reversible_encoding(s):
    # TODO: base64 round-trip; return decoded == s
    raise NotImplementedError

In [ ]:
# helpers (self-contained -- works in any run order)

def check(name, cond):

    print(f"{'PASS' if cond else 'FAIL'}  {name}"); return bool(cond)

def expect_error(fn, *a, exc=Exception):

    try: fn(*a)

    except exc: return True

    except Exception: return False

    return False



# Check - Part A

import hashlib

try:

    check('sha256 correct', sha256_hex(b'abc')==hashlib.sha256(b'abc').hexdigest())

    toks={secure_token(32) for _ in range(300)}

    check('tokens strong & unique', len(toks)==300 and all(len(t)>=32 for t in toks))

    check('base64 is reversible', is_reversible_encoding(b'secret-value') is True)

except NotImplementedError:

    print('TODO: implement the three functions above')


## Part B - Symmetric encryption done right (AEAD)  ·  A04:2025

ECB leaks repeated blocks and has no integrity. Use **AES-GCM** with a fresh nonce per message. *(needs `cryptography`)*

In [ ]:
try:
    from demo_symmetric import ecb_demo, gcm_demo, nonce_reuse_demo
    ecb_demo(); gcm_demo(); nonce_reuse_demo()
except ModuleNotFoundError:
    print("install first:  pip install cryptography")

**B.3 Your code**

In [ ]:
import os

def ecb_leaks(block):
    # TODO: AES-ECB on block+block; return ct[:16]==ct[16:32]
    raise NotImplementedError

def aead_encrypt(key, plaintext, aad=b''):
    # TODO: AESGCM; fresh 12-byte nonce; return nonce+ciphertext
    raise NotImplementedError

def aead_decrypt(key, blob, aad=b''):
    # TODO: split nonce, AESGCM.decrypt (raises on tamper)
    raise NotImplementedError

In [ ]:
# helpers (self-contained -- works in any run order)

def check(name, cond):

    print(f"{'PASS' if cond else 'FAIL'}  {name}"); return bool(cond)

def expect_error(fn, *a, exc=Exception):

    try: fn(*a)

    except exc: return True

    except Exception: return False

    return False



# Check - Part B

import os

try:

    check('ecb leaks', ecb_leaks(b'YELLOW SUBMARINE') is True)

    k=os.urandom(32); blob=aead_encrypt(k, b'hello', b'ctx')

    check('aead round-trip', aead_decrypt(k, blob, b'ctx')==b'hello')

    bad=bytearray(blob); bad[-1]^=1

    check('tamper rejected', expect_error(aead_decrypt, k, bytes(bad), b'ctx'))

    check('fresh nonce each call', aead_encrypt(k,b'x')[:12]!=aead_encrypt(k,b'x')[:12])

except NotImplementedError:

    print('TODO: implement the three functions above')

except ModuleNotFoundError:

    print("install first:  pip install cryptography")


## Part C - Integrity, MACs & key derivation  ·  A04:2025

A **MAC** detects tampering; compare it in **constant time**. Derive one key per purpose with a KDF - never reuse a key for two jobs.

In [ ]:
from demo_integrity import mac_demo, compare_demo, kdf_demo
mac_demo(); compare_demo(); kdf_demo()

**C.3 Your code**

In [ ]:
import hmac, hashlib

def hmac_sha256(key, msg):
    # TODO: hmac.new(key, msg, sha256).hexdigest()
    raise NotImplementedError

def verify_mac(key, msg, tag):
    # TODO: recompute, hmac.compare_digest(...)
    raise NotImplementedError

def derive_keys(master, labels=('enc','mac'), length=32):
    # TODO: HKDF-style per-label keys (dict label->bytes)
    raise NotImplementedError

In [ ]:
# helpers (self-contained -- works in any run order)

def check(name, cond):

    print(f"{'PASS' if cond else 'FAIL'}  {name}"); return bool(cond)

def expect_error(fn, *a, exc=Exception):

    try: fn(*a)

    except exc: return True

    except Exception: return False

    return False



# Check - Part C

try:

    t=hmac_sha256(b'k', b'm')

    check('mac verifies', verify_mac(b'k', b'm', t) is True)

    check('tampered mac fails', verify_mac(b'k', b'm', t[:-1]+('0' if t[-1]!='0' else '1')) is False)

    ks=derive_keys(b'master', ('enc','mac'), 32)

    check('keys distinct', ks['enc']!=ks['mac'] and len(ks['enc'])==32)

except NotImplementedError:

    print('TODO: implement the three functions above')


## Part D - Key management & damage reduction  ·  A04:2025

Never hard-code keys; separate keys per tenant/purpose; rotate with versioning so old data still decrypts.

In [ ]:
from demo_keymgmt import scan_demo, tenant_demo, rotation_demo
scan_demo(); tenant_demo(); rotation_demo()

**D.3 Your code**

In [ ]:
import re, hmac, hashlib, secrets

def find_hardcoded_secrets(source):
    # TODO: return stripped lines assigning a literal to key/secret/password/token/api_key
    raise NotImplementedError

def tenant_key(master, tenant_id):
    # TODO: HMAC(master, tenant_id) -> bytes
    raise NotImplementedError

def rotate(keyring):
    # TODO: add new v{n+1} key, keep old, return new id
    raise NotImplementedError

In [ ]:
# helpers (self-contained -- works in any run order)

def check(name, cond):

    print(f"{'PASS' if cond else 'FAIL'}  {name}"); return bool(cond)

def expect_error(fn, *a, exc=Exception):

    try: fn(*a)

    except exc: return True

    except Exception: return False

    return False



# Check - Part D

import os, secrets

try:

    src='API_KEY = "sk_live_1"\ndb_password = "hunter2"\ntimeout = 30\n'

    hits=find_hardcoded_secrets(src)

    check('secrets flagged', any('API_KEY' in h for h in hits) and any('db_password' in h for h in hits))

    check('timeout not flagged', not any('timeout' in h for h in hits))

    m=os.urandom(32)

    check('tenants isolated', tenant_key(m,'A')!=tenant_key(m,'B') and tenant_key(m,'A')==tenant_key(m,'A'))

    kr={'v1':secrets.token_bytes(32)}; nid=rotate(kr)

    check('rotation keeps old', nid!='v1' and 'v1' in kr and nid in kr)

except NotImplementedError:

    print('TODO: implement the three functions above')


## Done?

When every check prints `PASS`, copy your functions into `student_solutions.py` and run the full suite:

```bash
python -m pytest test_solutions.py -v
```